In [1]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

In [2]:
data = [
    ['bread', 'milk', 'nutmeg', 'onion', 'chips'],
    ['bread', 'chicken', 'chips', 'milk'],
    ['bread', 'chicken', 'eggs', 'nutmeg'],
    ['chips', 'eggs', 'onion'],
    ['chicken', 'milk', 'yogurt'],
    ['bread', 'eggs', 'nutmeg'],
    ['onion', 'milk', 'yogurt'],
    ['chips', 'milk']
]

In [3]:
te = TransactionEncoder()

In [4]:
te_array = te.fit_transform(data)
te_array

array([[ True, False,  True, False,  True,  True,  True, False],
       [ True,  True,  True, False,  True, False, False, False],
       [ True,  True, False,  True, False,  True, False, False],
       [False, False,  True,  True, False, False,  True, False],
       [False,  True, False, False,  True, False, False,  True],
       [ True, False, False,  True, False,  True, False, False],
       [False, False, False, False,  True, False,  True,  True],
       [False, False,  True, False,  True, False, False, False]])

In [5]:
te.columns_

['bread', 'chicken', 'chips', 'eggs', 'milk', 'nutmeg', 'onion', 'yogurt']

In [6]:
df = pd.DataFrame(te_array, columns=te.columns_)
df

,bread,chicken,chips,eggs,milk,nutmeg,onion,yogurt
0,True,False,True,False,True,True,True,False
1,True,True,True,False,True,False,False,False
2,True,True,False,True,False,True,False,False
3,False,False,True,True,False,False,True,False
4,False,True,False,False,True,False,False,True
5,True,False,False,True,False,True,False,False
6,False,False,False,False,True,False,True,True
7,False,False,True,False,True,False,False,False


In [7]:
# Apply Apriori algorithm to find frequent itemsets
frequent_items = apriori(df, min_support=0.3, use_colnames=True)
frequent_items

,support,itemsets
0,0.500,(bread)
1,0.375,(chicken)
2,0.500,(chips)
3,0.375,(eggs)
4,0.625,(milk)
5,0.375,(nutmeg)
6,0.375,(onion)
7,0.375,"(bread, nutmeg)"
8,0.375,"(chips, milk)"


In [8]:
# Generate association rules
rules = association_rules(frequent_items, metric='confidence', min_threshold= 0.2)
rules[['antecedents', 'consequents', 'support', 'confidence']]

,antecedents,consequents,support,confidence
0,(bread),(nutmeg),0.375,0.75
1,(nutmeg),(bread),0.375,1.00
2,(chips),(milk),0.375,0.75
3,(milk),(chips),0.375,0.60


### Interpretation.
- Antecedents: The item(s) on the left side of the rule (the “if” part).
- Consequents: The item(s) on the right side of the rule (the “then” part).
- Support: The proportion of transactions that contain both the antecedents and the consequents. For example, a support of 0.375 means 37.5% of all - transactions in the dataset include those items together.
- Confidence: Among the transactions where the antecedent items appear, the proportion that also contain the consequent items. A confidence of 1.0 (100%) means whenever the antecedent is purchased, the consequent is always purchased in those transactions.

#### Row 0

Rule: (nutmeg) → (bread) <br>
Support = 0.375: 37.5% of transactions contain both nutmeg and bread. <br>
Confidence = 1.00: In every transaction where nutmeg appears, bread also appears.
- Conclusion:
Whenever a customer buys nutmeg, they always buy bread as well (in this dataset). This could suggest placing nutmeg and bread closer together in a store or bundling them in promotions.

#### Row 1

Rule: (bread) → (nutmeg) <br>
Support = 0.375: 37.5% of transactions contain both bread and nutmeg.<br>
Confidence = 0.75: In 75% of the transactions where bread appears, nutmeg also appears.
- Conclusion:
Most of the time when bread is purchased, nutmeg is also purchased, but not always. This is a slightly weaker rule than the first one because it does not happen every time.

---
Market Basket Analysis helps reveal patterns in customer transactions by identifying which items tend to be purchased together. The Apriori algorithm is at the heart of this analysis. It works by:

- Finding Frequent Itemsets: It scans the dataset to find combinations of items that appear together frequently (above a specified support threshold).
- Generating Association Rules: From these itemsets, it then derives rules (e.g., if a customer buys item A, they are likely to also buy item B) using metrics such as support, confidence, and lift.

---

#### FP-Growth (short for Frequent Pattern Growth) is an efficient algorithm for frequent itemset mining in large datasets. It serves as an alternative to the Apriori algorithm but often outperforms Apriori by avoiding the costly candidate generation step.

In [9]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

# Sample transactional data (same as before)
data = [
    ['bread', 'milk', 'nutmeg', 'onion', 'chips'],
    ['bread', 'chicken', 'chips', 'milk'],
    ['bread', 'chicken', 'eggs', 'nutmeg'],
    ['chips', 'eggs', 'onion'],
    ['chicken', 'milk', 'yogurt'],
    ['bread', 'eggs', 'nutmeg'],
    ['onion', 'milk', 'yogurt'],
    ['chips', 'milk']
]

# Convert the data into a one-hot encoded DataFrame
te = TransactionEncoder()
te_array = te.fit_transform(data)  # Equivalent to fit() then transform()
df = pd.DataFrame(te_array, columns=te.columns_)

In [10]:
frequent_items = fpgrowth(df, min_support=0.3, use_colnames=True)
frequent_items

,support,itemsets
0,0.625,(milk)
1,0.500,(chips)
2,0.500,(bread)
3,0.375,(onion)
4,0.375,(nutmeg)
5,0.375,(chicken)
6,0.375,(eggs)
7,0.375,"(chips, milk)"
8,0.375,"(bread, nutmeg)"


In [11]:
rules = association_rules(frequent_items, metric="confidence", min_threshold=0.2)
rules[['antecedents', 'consequents', 'support', 'confidence']]

,antecedents,consequents,support,confidence
0,(chips),(milk),0.375,0.75
1,(milk),(chips),0.375,0.60
2,(bread),(nutmeg),0.375,0.75
3,(nutmeg),(bread),0.375,1.00


In [12]:
import graphviz

In [13]:
graph = graphviz.Digraph('Association_Rules', format='png')

for index, row in rules.iterrows():
    antecedent_str = ', '.join(list(row['antecedents']))
    consequent_str = ', '.join(list(row['consequents']))

    # Create a label showing support and confidence (optional)
    edge_label = f"sup={row['support']}, conf={row['confidence']}"

    # Add an edge to the graph
    graph.edge(antecedent_str, consequent_str, label=edge_label)

# Render the graph (this creates 'Association_Rules.png' and tries to open it)
graph.render('Association_Rules', view=True)


ExecutableNotFound: failed to execute WindowsPath('dot'), make sure the Graphviz executables are on your systems' PATH